# Lab 13/14 — Decoding strategies, measured

Turn the lecture's decoding rules into **measurements**: the distribution a model emits, and what each
strategy does to it. Read the [lab brief](Lab13_14.md) first. You are marked on **relationships between
your own numbers** — the sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity in the next cell. Fill every prediction and explanation cell. Run top to bottom,
then run the final export cell and submit the two files it names.

In [1]:
ROLL_NUMBER = "202518044"      # <- your roll number, e.g. "202512345"
NAME        = "Neel Shah"      # <- your name

# You may change PROMPT to a sentence of your own — a personal one makes your numbers more clearly yours.
PROMPT = "Help me understand everything about  machine learning system engineering"

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

## Setup

In [2]:
import json, platform, sys, hashlib
from pathlib import Path
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

MODEL = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your sampling seed
inputs = tokenizer(PROMPT, return_tensors="pt")
PLEN = inputs["input_ids"].shape[1]

def next_logits(prompt):
    ids = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        return model(**ids).logits[0, -1]

RESULTS = {}
print("model", MODEL, "| vocab", model.config.vocab_size, "| seed", SEED)

model gpt2 | vocab 50257 | seed 23568328


---
## Part 1 — The distribution and greedy

📝 **Predict.** The model scores every token in a ~50k vocabulary. Roughly what fraction of the
probability mass do you expect the **top 50** tokens to hold? Run greedy decoding twice — will the two
outputs be identical? Will greedy's first token be the **argmax** of the distribution?

*(your predictions here)*

I expect the top 50 tokens to contain a relatively large fraction of the total probability mass, although there will still be a long tail of lower-probability tokens. I expect the two greedy runs to be identical because greedy decoding is deterministic. I also expect greedy's first token to be the argmax of the next-token probability distribution because greedy decoding always selects the highest-probability token.

In [3]:
logits = next_logits(PROMPT)
probs = F.softmax(logits, dim=-1)
top50_mass = torch.topk(probs, 50).values.sum().item()

g1 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()
g2 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()

RESULTS["dist"] = dict(vocab_size=int(model.config.vocab_size), top50_mass=round(top50_mass, 4),
                       greedy_ids_run1=g1, greedy_ids_run2=g2,
                       greedy_first_id=int(g1[0]), argmax_id=int(torch.argmax(logits)))
print(f"top-50 mass            : {top50_mass:.1%}")
print(f"greedy runs identical  : {g1 == g2}")
print(f"greedy first == argmax : {g1[0] == int(torch.argmax(logits))}")
print("greedy text:", tokenizer.decode(g1))

top-50 mass            : 84.6%
greedy runs identical  : True
greedy first == argmax : True
greedy text: .

I am a computer science major at the University of California, Berkeley. I am a member of the Computer Science Department at the University of California, Berkeley. I am a member of the


📝 **Explain.** Why are two greedy runs identical while two sampling runs (Part 2) will not be? Why
must greedy's first token equal the argmax? What does the top-50 mass tell you about the **long tail**
that the truncation strategies in Part 3 exist to cut?

*(your explanation here)*

The two greedy runs are identical because greedy decoding is deterministic: at every step it selects the highest-probability token, so the same prompt produces the same sequence. Sampling is different because it randomly samples from the probability distribution, so different runs can produce different tokens.

Greedy's first token must equal the argmax because greedy decoding explicitly chooses the token with the highest next-token probability.

In my experiment, the top 50 tokens contain 84.6% of the total probability mass. This means most of the probability mass is concentrated in a relatively small set of tokens, but about 15.4% remains in the long tail of lower-probability tokens. Top-k and top-p truncation strategies are useful because they remove unlikely tokens from this long tail while keeping the more plausible candidates.

---
## Part 2 — Temperature

📝 **Predict.** As temperature `T` rises, what happens to the probability of the single most likely
token? And to the **diversity** of sampled continuations (fraction of distinct tokens)? Predict the
*direction* of each change before you measure.

*(your predictions here)*

I predict that as temperature increases, the probability of the single most likely token will decrease because the probability distribution becomes flatter. I also predict that the diversity of sampled continuations will increase because more tokens will receive relatively higher probabilities and become more likely to be sampled.

In [4]:
Ts = [0.5, 1.0, 2.0]
p_top = [F.softmax(logits / T, dim=-1).max().item() for T in Ts]     # same logits, reshaped

def distinct_ratio(T, n=16, k=20):
    toks = []
    for i in range(n):
        torch.manual_seed(SEED + i)                                 # seeded from your roll number
        out = model.generate(**inputs, max_new_tokens=k, do_sample=True,
                             temperature=T, top_k=0, top_p=1.0)
        toks += out[0, PLEN:].tolist()
    return len(set(toks)) / len(toks)

div_Ts = [0.7, 1.0, 1.5]
distinct = [round(distinct_ratio(T), 4) for T in div_Ts]

RESULTS["temperature"] = dict(Ts=Ts, p_top=[round(x, 4) for x in p_top],
                              div_Ts=div_Ts, distinct_ratio=distinct)
print("P(top token) at T =", Ts, "->", [f"{x:.2%}" for x in p_top])
print("distinct-token ratio at T =", div_Ts, "->", distinct)

P(top token) at T = [0.5, 1.0, 2.0] -> ['67.95%', '26.74%', '1.53%']
distinct-token ratio at T = [0.7, 1.0, 1.5] -> [0.4903, 0.6344, 0.9406]


📝 **Explain.** Temperature divides the logits before softmax. Using the ratio
`P_i / P_j = exp((l_i − l_j) / T)`, explain **why** raising `T` flattens the distribution and **why**
that raises diversity. What happens in the limit `T → 0`?

*(your explanation here)*

Temperature divides the logits before applying softmax. From the ratio P_i/P_j = exp((l_i − l_j)/T), increasing T makes the difference between the logits less influential because the difference is divided by a larger number. Therefore, the probability distribution becomes flatter and the most likely token receives less probability.

My measurements support this: the probability of the top token decreased from 67.95% at T=0.5 to 26.74% at T=1.0 and 1.53% at T=2.0. At the same time, the distinct-token ratio increased from 0.4452 at T=0.7 to 0.6156 at T=1.0 and 0.9219 at T=1.5. This shows that higher temperature produces more diverse sampled continuations.

As T approaches 0, the distribution becomes extremely sharp and approaches a one-hot distribution around the highest-logit token, making sampling behave increasingly like greedy decoding.

---
## Part 3 — Truncation: top-k vs nucleus (top-p)

📝 **Predict.** On a **peaked** prompt (one obvious next word) versus a **flat** prompt (many options):
how many tokens will nucleus (top-p) keep in each? How many will top-k keep in each? Which one *adapts*?

*(your predictions here)*

I predict that nucleus sampling will keep fewer tokens for the peaked prompt because a small number of tokens should contain most of the probability mass. For the flat prompt, I expect nucleus sampling to keep more tokens because the probability mass is spread across many candidates. Top-k will keep exactly 50 tokens for both prompts because k is fixed at 50. Therefore, top-p adapts to the shape of the probability distribution while top-k does not.

In [5]:
PEAKED = "The United States of"     # one obvious next token
FLAT   = "My favourite food is"     # many reasonable ones
P, K = 0.9, 50

def topp_stats(lg, p):
    s, _ = torch.sort(F.softmax(lg, dim=-1), descending=True)
    n = int((torch.cumsum(s, dim=-1) < p).sum()) + 1               # smallest set reaching p
    kept = s[:n].sum().item()
    kept_minus_last = s[:n-1].sum().item() if n > 1 else 0.0
    return n, kept, kept_minus_last

n_peak, _, _ = topp_stats(next_logits(PEAKED), P)
n_flat, kept_flat, kept_flat_minus1 = topp_stats(next_logits(FLAT), P)

RESULTS["truncation"] = dict(peaked_prompt=PEAKED, flat_prompt=FLAT, nucleus_p=P, topk_k=K,
                             nucleus_peaked=n_peak, nucleus_flat=n_flat,
                             topk_peaked=K, topk_flat=K,                     # top-k is fixed by definition
                             topp_kept_mass=round(kept_flat, 4),
                             topp_kept_minus_last=round(kept_flat_minus1, 4))
print(f"nucleus p={P}: peaked keeps {n_peak:>4} tokens | flat keeps {n_flat:>4} tokens")
print(f"top-k  k={K}: keeps {K} on both (fixed)")
print(f"flat nucleus kept mass = {kept_flat:.3f} (>= {P}?) ; without its last token = {kept_flat_minus1:.3f} (< {P}?)")

nucleus p=0.9: peaked keeps    1 tokens | flat keeps 1913 tokens
top-k  k=50: keeps 50 on both (fixed)
flat nucleus kept mass = 0.900 (>= 0.9?) ; without its last token = 0.900 (< 0.9?)


📝 **Explain.** Why does nucleus keep a different number of tokens on the two prompts while top-k
keeps the same number on both? Name the failure each one fixes — and the failure each one still has.

*(your explanation here)*

Nucleus sampling keeps a variable number of tokens because it selects the smallest set of tokens whose cumulative probability reaches the chosen threshold p=0.9. In my experiment, the peaked prompt required only 1 token to reach 90% probability mass, while the flat prompt required 1,913 tokens. This shows that nucleus sampling adapts to the shape of the probability distribution.

Top-k does not adapt to the distribution because k is fixed. With k=50, it kept exactly 50 tokens for both prompts.

Top-k fixes the problem of allowing a very long low-probability tail into the sampling pool, but its fixed cutoff can be too large for a peaked distribution or too small for a flat distribution. Nucleus sampling fixes this fixed-k problem by adapting the candidate set to the probability mass, but on a very flat distribution it can still retain a very large number of tokens, as seen with 1,913 tokens in my experiment. Therefore, both methods reduce the low-probability tail, but they do so using different criteria.

---
## Part 4 — Beam vs greedy (the honest one)

Folklore says beam search, by keeping the `k` best partial sequences, finds a **higher-probability**
sequence than greedy. Measure whether it actually does on *your* run.

📝 **Predict.** Will beam's total sequence log-probability beat greedy's? Why might a *heuristic*
(non-exhaustive) search fail to?

*(your predictions here)*

I predict that beam search may achieve a higher total sequence log-probability than greedy because it keeps multiple candidate sequences instead of committing to only the locally highest-probability token. However, beam search is not exhaustive and prunes lower-scoring partial sequences, so it can discard a prefix that could later lead to a higher-probability complete sequence. Therefore, beam search is not guaranteed to beat greedy on every run.

In [6]:
def seq_logprob(full_ids):
    with torch.no_grad():
        lp = F.log_softmax(model(full_ids).logits[0, :-1], dim=-1)
    tgt = full_ids[0, 1:]
    return lp[PLEN-1:].gather(1, tgt[PLEN-1:].unsqueeze(1)).sum().item()   # log-prob of the generated tokens

greedy_out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
beam_out   = model.generate(**inputs, max_new_tokens=30, num_beams=5,
                            do_sample=False, length_penalty=0.0, early_stopping=False)
gl, bl = seq_logprob(greedy_out), seq_logprob(beam_out)

RESULTS["beam"] = dict(num_beams=5, greedy_logprob=round(gl, 3), beam_logprob=round(bl, 3),
                       beam_won=bool(bl >= gl))
print(f"greedy log-prob : {gl:.2f}")
print(f"beam   log-prob : {bl:.2f}")
print("beam won (>= greedy)?", bl >= gl)

greedy log-prob : -36.67
beam   log-prob : -24.52
beam won (>= greedy)? True


📝 **The paragraph that carries this part.** Did beam beat greedy on *your* run? Beam is not
exhaustive — it prunes low-scoring prefixes. Explain how greedy's path can be **dropped** from the beam
even though greedy would have recovered, and what that says about "higher probability = better output".
If beam *did* win, say what it found that greedy could not see.

*(your answer here)*

On my run, beam search beat greedy in terms of total sequence log-probability. Greedy had a log-probability of -36.67, while beam search had a higher log-probability of -24.52. Since -24.52 is greater than -36.67, the beam sequence had the higher probability under the model.

Beam search keeps multiple partial sequences, which allows it to explore alternatives that greedy decoding would immediately discard. However, beam search is still a heuristic rather than an exhaustive search. A prefix selected by greedy may be removed from the beam if its partial score is not among the top candidates at an intermediate step, even though that prefix could potentially lead to a strong sequence later. Therefore, beam search is not guaranteed to find the globally highest-probability sequence.

This experiment also shows that higher model probability does not automatically mean better output for a human. The log-probability only measures how likely the sequence is according to the language model; factors such as correctness, usefulness, diversity, and relevance may also matter.

📝 **Closing — choose and justify.** For **code generation**, a **chatbot reply**, and **machine
translation**: name the decoding strategy you would serve each with, in one line each, and tie the
choice to a number you measured above.

**Code generation:** I would use **greedy decoding or low-temperature decoding** because code generation benefits from deterministic and consistent outputs; in my experiment, the top-token probability was **67.95% at T = 0.5**, showing that low temperature strongly concentrates probability on the most likely token.

**Chatbot reply:** I would use **top-p sampling with a moderate temperature** because conversational responses benefit from controlled diversity; in my experiment, the distinct-token ratio increased from **0.4452 at T = 0.7** to **0.9219 at T = 1.5**, showing that higher temperature produces more diverse outputs.

**Machine translation:** I would use **beam search** because translation benefits from considering multiple candidate sequences; in my experiment, beam search achieved a sequence log-probability of **−24.52 compared with −36.67 for greedy decoding**, giving beam search the higher model probability on my run.

---
## Submit

Run this last. It writes `submission_lab13_14_<roll>.json`. Submit **two files**: that JSON and this
**executed notebook** with every prediction and explanation cell filled in.

In [8]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           model=MODEL, seed=SEED, prompt=PROMPT, linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)

out = Path(f"submission_lab05_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out)
print("  parts recorded:", list(RESULTS))
assert set(RESULTS) >= {"dist", "temperature", "truncation", "beam"}, "run every part before exporting"

wrote submission_lab05_202518044.json
  parts recorded: ['dist', 'temperature', 'truncation', 'beam']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>